<h1 style="color:#1f77b4; font-family:'Times New Roman';">
<b>Hyperparameter Tuning</b>
</h1>
<div style="font-family:'Times New Roman';">
Every model has knobs you set before training, the hyperparameters, like the number of trees or the max depth. The defaults are ok but rarely the best. Tuning is just searching for a good combination of these knobs, and we use cross validation to score each combination fairly. Two common ways, grid search and random search.
</div>

In [1]:
import numpy as np
from scipy.stats import randint

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [2]:
data = load_breast_cancer()
X, y = data.data, data.target

# keep a test set aside that the search never sees
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('train:', Xtr.shape, ' test:', Xte.shape)

train: (455, 30)  test: (114, 30)


## Grid search

Grid search just tries every single combination in a grid of values you give it, and scores each one with cross validation. Its thorough but can get slow if the grid is big, because it tries literally everything.

In [3]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, None],
    'min_samples_split': [2, 5],
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, n_jobs=-1)
grid.fit(Xtr, ytr)

print('best params:', grid.best_params_)
print('best cv score:', round(grid.best_score_, 4))

best params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
best cv score: 0.9604


## Random search

Instead of trying everything, random search just picks a fixed number of random combinations from the ranges you give. Sounds worse but it often finds something just as good way faster, specially when only a couple of the knobs actually matter.

In [5]:
param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': [3, 5, 7, None],
    'min_samples_split': randint(2, 10),
}

rand = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_dist,
                          n_iter=15, cv=5, random_state=42, n_jobs=-1)
rand.fit(Xtr, ytr)

print('best params:', rand.best_params_)
print('best cv score:', round(rand.best_score_, 4))

best params: {'max_depth': 7, 'min_samples_split': 5, 'n_estimators': 142}
best cv score: 0.956


<h2 style="color:#1f77b4; font-family:'Times New Roman';">
<b>Evaluating the tuned model properly</b>
</h2>
<div style="font-family:'Times New Roman';">
The cross validation score above was measured on the training data during the search. The honest test is the held out test set the search never touched. Here is where the metrics from the first notebook come back, confusion report and auc.
</div>

In [6]:
best = grid.best_estimator_
test_pred = best.predict(Xte)
test_proba = best.predict_proba(Xte)[:, 1]

print(classification_report(yte, test_pred, target_names=data.target_names))
print('test AUC:', round(roc_auc_score(yte, test_proba), 4))

              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

test AUC: 0.9931


### what i learned here

- hyperparameters are the knobs you set before training, and tuning searches for good ones
- grid search tries every combination, random search samples a few and is usually faster
- both use cross validation inside, so each combo gets a fair, steady score
- always keep a separate test set for the final check, because the search already peeked at the cv folds

putting the three notebooks together, this is basically the proper way to judge any model from now on, good metrics, cross validation, and a tuned model checked on data it never saw.